# 逐频各向异性复现：从天空功率到检测统计量

这份 notebook 是项目的主入口，直观展示论文数值模拟的每一步。当前范围仅为**点源模拟**，不含 holodeck。

绿色路线是已经闭环的物理步骤；橙色路线需要论文未公开的逐脉冲星对 PFOS 误差，因此只作演示，不能标成图 4 的忠实复现。现有确定性 CW leakage 工作放在最后，保持与论文随机背景模型分离。

## 复现路线

1. 锁定论文参数与目标数值。
2. 建立 NANOGrav 68/67 颗脉冲星阵列。
3. 构造 Earth-term 像素响应矩阵。
4. 用各向同性天空恢复 Hellings--Downs 曲线。
5. 在 48 个候选方向注入点源，并精确控制 C1/C0。
6. 生成未抽样的理论脉冲星对相关量。
7. 用平方根球谐似然比和逐像素 radiometer 重建。
8. 独立 null 模拟确定阈值，再与论文图 4 检测率比较。
9. 基线复现通过后，才加入 on-grid/off-grid CW leakage。

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ.setdefault('MPLCONFIGDIR', '/tmp/leakage-matplotlib')

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

from leakage.paper import (
    PAPER_ANISOTROPY_LEVELS, PAPER_LMAX, PAPER_MEDIAN_PAIR_SIGMA,
    PAPER_NSIDE, PAPER_OBSERVATION_SPAN_YR, PAPER_PIXEL_DETECTION_RATE,
    PAPER_REPORTED_FREQUENCIES_NHZ, PAPER_SNR_DETECTION_RATE,
    angular_power_ratio, candidate_uniform_pixels, correlations_from_sky,
    fourier_frequencies_nhz, isotropic_plus_point_map, load_ng15_positions,
)
from leakage.response import hellings_downs, pair_indices, pixel_response_matrix
from leakage.statistics import anisotropy_likelihood_ratio, radiometer_map

plt.rcParams.update({'figure.figsize': (8, 4.8), 'font.size': 11})
print('项目目录:', PROJECT_ROOT)

: 

## 1. 论文参数：先锁定，再计算

论文使用 `Nside=8`、功率 `lmax=6`、16.03 年观测跨度、最低五个频段和三种点源各向异性强度。注意：论文列出的频率是 1.98 nHz 的整数倍，与精确的 k/T 有轻微差异。

In [ ]:
exact_frequencies = fourier_frequencies_nhz()
print(f'Nside={PAPER_NSIDE}, power lmax={PAPER_LMAX}, T={PAPER_OBSERVATION_SPAN_YR} yr')
print('论文列值 [nHz]:', PAPER_REPORTED_FREQUENCIES_NHZ)
print('精确 k/T [nHz]: ', np.round(exact_frequencies, 6))
print('两者差值 [nHz]: ', np.round(PAPER_REPORTED_FREQUENCIES_NHZ - exact_frequencies, 6))
print('C1/C0:', PAPER_ANISOTROPY_LEVELS)
print('论文五频段 sigma 中位数:', PAPER_MEDIAN_PAIR_SIGMA)

## 2. NANOGrav 68/67 颗脉冲星

论文在信号生成处写 68 颗；标准 NG15 GWB 分析去掉观测不足三年的 J0614−3329，使用 67 颗。本项目保留两条分支，下面默认展示论文逐字的 68 颗分支。

In [ ]:
names67, vectors67 = load_ng15_positions()
names68, vectors68 = load_ng15_positions(include_short_baseline=True)
print(f'标准分支: {len(names67)} 颗, {len(names67)*(len(names67)-1)//2} 对')
print(f'论文分支: {len(names68)} 颗, {len(names68)*(len(names68)-1)//2} 对')
print('第 68 颗:', names68[-1])

ra = np.arctan2(vectors68[:, 1], vectors68[:, 0])
dec = np.arcsin(vectors68[:, 2])
fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111, projection='mollweide')
ax.scatter(-ra[:-1], dec[:-1], s=22, label='NG15 standard 67')
ax.scatter(-ra[-1], dec[-1], marker='*', s=130, color='tab:red', label='J0614-3329')
ax.grid(alpha=0.35)
ax.legend(loc='lower left')
ax.set_title('NANOGrav 15-year pulsar sky positions')
plt.show()

## 3. 天空功率到脉冲星对相关量

像素响应采用论文式 (19)：每个像素的 plus/cross 天线响应乘积，乘以 3/(2 Npix)。这里只保留 Earth term；地图坐标是引力波传播方向，物理源方向是其对跖点。

In [ ]:
response = pixel_response_matrix(vectors68, nside=PAPER_NSIDE)
print('响应矩阵形状:', response.shape, '= (脉冲星对, 天空像素)')
print('内存约:', round(response.nbytes / 1024**2, 1), 'MiB')

## 4. Gate A：各向同性天空必须恢复 HD 曲线

这是最重要的前向模型检查。图像相似不够；像素积分必须数值上逼近解析 Hellings--Downs 相关。

In [ ]:
isotropic_sky = np.ones(hp.nside2npix(PAPER_NSIDE))
rho_isotropic = correlations_from_sky(response, isotropic_sky)
rho_hd = hellings_downs(vectors68)
left, right = pair_indices(len(vectors68))
separation = np.arccos(np.clip(np.sum(vectors68[left] * vectors68[right], axis=1), -1, 1))
order = np.argsort(separation)

plt.scatter(np.rad2deg(separation), rho_isotropic, s=5, alpha=0.35, label='Nside=8 pixel integral')
plt.plot(np.rad2deg(separation[order]), rho_hd[order], color='black', lw=1.6, label='analytic HD')
plt.xlabel('Pulsar separation [deg]')
plt.ylabel('Normalized correlation')
plt.legend()
plt.show()
print('最大绝对误差:', np.max(np.abs(rho_isotropic - rho_hd)))
print('RMS 误差:', np.sqrt(np.mean((rho_isotropic - rho_hd)**2)))

## 5. 48 个点源方向

论文只说“48 个均匀分布的像素”，没有公布索引。当前可证伪候选方案是：取 Nside=2 的全部 48 个像素中心，再投影到 Nside=8。它是敏感性分析分支，不冒充作者原坐标。

In [ ]:
candidate_pixels = candidate_uniform_pixels(PAPER_NSIDE)
theta48, phi48 = hp.pix2ang(PAPER_NSIDE, candidate_pixels)
print('方向数:', len(candidate_pixels), '唯一像素数:', len(np.unique(candidate_pixels)))
fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111, projection='mollweide')
ax.scatter(-phi48, np.pi/2 - theta48, s=35, color='tab:orange', label='48 candidate directions')
ax.scatter(-ra, dec, s=10, color='tab:blue', alpha=0.65, label='pulsars')
ax.grid(alpha=0.35)
ax.legend(loc='lower left')
plt.show()

## 6. 点源注入：控制 C1/C0

在均匀背景的一个像素中加入功率，并在实际 HEALPix 网格上求解点源幅度，使角功率谱严格满足目标 C1/C0。全天平均功率随后归一为 1；归一化不改变 C1/C0。

In [ ]:
INJECTION_DIRECTION = 0
TARGET_RATIO = 0.3
injection_pixel = int(candidate_pixels[INJECTION_DIRECTION])
point_sky = isotropic_plus_point_map(injection_pixel, TARGET_RATIO, nside=PAPER_NSIDE)
print('注入像素:', injection_pixel)
print('恢复的 C1/C0:', angular_power_ratio(point_sky))
print('天空平均功率:', point_sky.mean(), '最大像素功率:', point_sky.max())
hp.mollview(point_sky, title='Isotropic background + one injected point pixel', unit='relative power')
hp.graticule(alpha=0.35)
plt.show()

## 7. 理论相关量：各向异性如何偏离 HD

论文点源实验不对相关量再次抽样，而是直接使用天空功率与天线响应合成的理论相关值。各向异性使相关量不再只是脉冲星夹角的单值函数。

In [ ]:
rho_point = correlations_from_sky(response, point_sky)
delta_rho = rho_point - rho_isotropic
plt.scatter(np.rad2deg(separation), rho_isotropic, s=6, alpha=0.25, label='isotropic')
sc = plt.scatter(np.rad2deg(separation), rho_point, c=delta_rho, s=8, cmap='coolwarm', label='point injection')
plt.colorbar(sc, label='departure from isotropy')
plt.xlabel('Pulsar separation [deg]')
plt.ylabel('Pair correlation')
plt.legend()
plt.show()

## 8. 两种重建统计量

- **似然比 SNR**：比较 lmax=6 平方根球谐模型和各向同性模型。平方根场只展开到 l=3，平方后功率可到 l=6。
- **像素 radiometer**：逐像素单独拟合功率，用于局部热点上限。

> 下面为了直观看流程，把所有脉冲星对的误差暂设为论文第一频段报告的中位数 11.76。论文没有公布完整 sigma_ab，因此该结果只是管线演示，不是图 4 复现值。

In [ ]:
sigma_demo = np.full(len(rho_point), PAPER_MEDIAN_PAIR_SIGMA[0])
likelihood_result = anisotropy_likelihood_ratio(
    response, rho_point, sigma_demo, power_lmax=PAPER_LMAX, n_starts=4, seed=20250902
)
radio, radio_sigma = radiometer_map(response, rho_point, sigma_demo, normalize_mean=True)
print('各向同性 chi2:', likelihood_result.isotropic_chi2)
print('各向异性 chi2:', likelihood_result.anisotropic_fit.chi2)
print('演示 SNR:', likelihood_result.snr)
print('注意：这是统一中位误差演示，不是论文检测判定。')

fig = plt.figure(figsize=(13, 7))
hp.mollview(point_sky, fig=fig.number, sub=(2, 2, 1), title='Injected map', min=0)
hp.mollview(likelihood_result.anisotropic_fit.sky_map, fig=fig.number, sub=(2, 2, 2), title='sqrt-SH fit, lmax=6', min=0)
hp.mollview(radio, fig=fig.number, sub=(2, 2, 3), title='pixel radiometer')
hp.mollview(radio_sigma, fig=fig.number, sub=(2, 2, 4), title='radiometer uncertainty', min=0)
plt.show()

## 9. Null 校准：阈值不能看着注入结果调

正式顺序必须是：

1. 只生成各向同性 null 样本。
2. 固定随机种子、样本数和 3 sigma 定义。
3. 得到 SNR 阈值及逐像素阈值。
4. 冻结阈值后才打开点源注入结果。
5. 同时报告逐像素假阳性率和全天 look-elsewhere 后的假阳性率。

下面只用少量样本画出 null 污染长什么样。3 sigma 单侧分位数为 0.99865；可靠估计尾部至少需要数万次，而不是下面的预览次数。

In [ ]:
PREVIEW_NULL = 40
rng = np.random.default_rng(20250903)
null_maps = []
for _ in range(PREVIEW_NULL):
    rho_null = rho_isotropic + rng.normal(0.0, sigma_demo)
    null_map, _ = radiometer_map(response, rho_null, sigma_demo, normalize_mean=False)
    null_maps.append(null_map)
null_maps = np.asarray(null_maps)
preview_threshold = np.quantile(null_maps, 0.95, axis=0)
print('单侧 3 sigma 分位数:', norm.cdf(3.0))
print('本格仅显示 40 次 null 的 95% 预览，不用于检测。')
hp.mollview(preview_threshold, title='Null contamination preview: 95% pointwise level')
hp.graticule(alpha=0.35)
plt.show()

## 10. 图 4 的正式验收目标

每个百分比对应 48 个天空方向中的成功次数。只有使用完整逐对逐频 PFOS 误差、冻结的 null 阈值和明确的 48 个方向后，才可以和这些目标比较。

In [ ]:
for row, ratio in enumerate(PAPER_ANISOTROPY_LEVELS):
    print(f'C1/C0={ratio:.1f}')
    print('  SNR 检测率 [%]: ', np.round(100 * PAPER_SNR_DETECTION_RATE[row], 2))
    print('  像素检测率 [%]:', np.round(100 * PAPER_PIXEL_DETECTION_RATE[row], 2))

## 11. 当前状态与下一关

已经通过：

- 单位向量、脉冲星对顺序和响应矩阵维度。
- 各向同性像素积分恢复 HD。
- 点像素响应与直接天线乘积一致。
- 点源地图精确达到 C1/C0 = 0.3/0.4/0.5。
- 平方根球谐与 radiometer 的最小相关量空间闭环。

尚未通过：

- 从正式 NG15 数据重算五频段完整 sigma_ab。
- 锁定论文 48 个像素索引、Monte Carlo 次数和 3 sigma 定义。
- 正式 null 假阳性率、覆盖率和 look-elsewhere 校准。
- 在 Monte Carlo 误差内重现图 4。

## 12. 复现之后：进入频率—天区联合 leakage

论文基线通过后，在完全相同的阵列、误差和阈值下依次加入：

1. 随机背景热点；
2. on-grid CW；
3. off-grid CW，频率写成 f0=(k+delta)/T；
4. CW 与各向同性 GWB 的混合。

最终比较三层信息：单频 sky map、CW 脉冲星对空间指纹、跨频泄漏形状。目标不是只说“有热点”，而是判断热点更像随机涨落还是一个 off-grid CW。